In [7]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
!wget $data

--2026-10-05 19:40:11--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv.1’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.009s  

2026-10-05 19:40:11 (66.2 MB/s) - ‘car_fuel_efficiency_2026.csv.1’ saved [635180/635180]



In [23]:
import pandas as pd
import numpy as np

In [17]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')

In [18]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [19]:
columns = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[columns]

In [20]:
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [21]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [22]:
df['horsepower'].median()

np.float64(254.0)

In [24]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [41]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0]) 
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    mse = ((y - y_pred) ** 2).mean()
    return np.sqrt(mse)

In [42]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

base_features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
df_train_features = df_train[base_features]
df_val_features = df_val[base_features]

In [43]:
X_train_0 = df_train_features.fillna(0).values
w0_0, w_0 = train_linear_regression(X_train_0, y_train)

X_val_0 = df_val_features.fillna(0).values
y_pred_0 = w0_0 + X_val_0.dot(w_0)

score_0 = rmse(y_val, y_pred_0)
score_0

np.float64(2.207036592817851)

In [44]:
mean_horsepower = df_train_features['horsepower'].mean()

X_train_mean = df_train_features.fillna(mean_horsepower).values
w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

X_val_mean = df_val_features.fillna(mean_horsepower).values
y_pred_mean = w0_mean + X_val_mean.dot(w_mean)

score_mean = rmse(y_val, y_pred_mean)
score_mean

np.float64(2.2040135462211143)

In [45]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    # Добавляем регуляризацию к диагонали матрицы
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]


In [46]:
X_train_0 = df_train_features.fillna(0).values
X_val_0 = df_val_features.fillna(0).values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train_0, y_train, r=r)
    y_pred = w0 + X_val_0.dot(w)
    score = rmse(y_val, y_pred)
    print(f"r = {r:<5} | RMSE = {round(score, 5)}")

r = 0     | RMSE = 2.20704
r = 0.01  | RMSE = 2.20666
r = 0.1   | RMSE = 2.21953
r = 1     | RMSE = 2.33645
r = 5     | RMSE = 2.39515
r = 10    | RMSE = 2.40506
r = 100   | RMSE = 2.41463


In [47]:
# Список всех seed значений для проверки
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmse_scores = []

# Базовые признаки
base_features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

for s in seeds:
    # 1. Задаем seed и перемешиваем индексы
    np.random.seed(s)
    idx = np.arange(len(df))
    np.random.shuffle(idx)
    
    # 2. Разбиваем датасет (60 / 20 / 20)
    df_train = df.iloc[idx[:n_train]].copy().reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train+n_val]].copy().reset_index(drop=True)
    
    # 3. Выделяем таргет
    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values
    
    # 4. Выделяем фичи и заполняем пропуски нулем
    X_train = df_train[base_features].fillna(0).values
    X_val = df_val[base_features].fillna(0).values
    
    # 5. Обучаем базовую модель без регуляризации
    # Используйте ту функцию, которую вы написали для Q3 (возвращающую w0 и w)
    w0, w = train_linear_regression(X_train, y_train)
    
    # 6. Предсказываем и считаем RMSE
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    
    # Сохраняем скор для этого seed
    rmse_scores.append(score)

# Считаем стандартное отклонение всех скоров
std_value = np.std(rmse_scores)
print(f"Стандартное отклонение (std): {round(std_value, 3)}")


Стандартное отклонение (std): 0.029


In [48]:
# 1. Выполняем разделение с seed=9
np.random.seed(9)
idx = np.arange(len(df))
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy().reset_index(drop=True)
df_val = df.iloc[idx[n_train:n_train+n_val]].copy().reset_index(drop=True)
df_test = df.iloc[idx[n_train+n_val:]].copy().reset_index(drop=True)

# 2. Объединяем тренировочный и валидационный датасеты
df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

# 3. Выделяем таргет
y_full_train = df_full_train.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

# 4. Заполняем пропуски нулем и забираем фичи
base_features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
X_full_train = df_full_train[base_features].fillna(0).values
X_test = df_test[base_features].fillna(0).values

# 5. Обучаем модель с r=0.001
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

# 6. Предсказываем на тесте и считаем финальный RMSE
y_pred = w0 + X_test.dot(w)
final_score = rmse(y_test, y_pred)

print(f"Финальный RMSE на тесте: {round(final_score, 3)}")


Финальный RMSE на тесте: 2.236
